In [4]:
import os
import shutil
import random
from tqdm import tqdm

# Define paths
source_dir = "model/img-dress_type"
train_dir = "model/dress_type/train"
val_dir = "model/dress_type/val"

# Make directories
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# 80% train, 20% validation
for cls in tqdm(os.listdir(source_dir)):
    cls_path = os.path.join(source_dir, cls)
    if not os.path.isdir(cls_path):
        continue

    images = os.listdir(cls_path)
    random.shuffle(images)

    split_idx = int(len(images) * 0.8)
    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(train_dir, cls, img))
    for img in val_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(val_dir, cls, img))


100%|██████████| 277/277 [00:33<00:00,  8.38it/s]


In [2]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# Paths
train_path = "dress_type/train"
val_path = "dress_type/val"

# Data transforms (augmentation + normalization)
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    "val": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

# Datasets and loaders
train_data = datasets.ImageFolder(train_path, transform=data_transforms["train"])
val_data   = datasets.ImageFolder(val_path, transform=data_transforms["val"])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=4)

# Model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(weights='IMAGENET1K_V2')
model.fc = nn.Linear(model.fc.in_features, len(train_data.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Training loop
best_acc = 0
for epoch in range(10):  # increase epochs for better accuracy
    model.train()
    train_loss, correct, total = 0, 0, 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/10"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_correct / val_total
    print(f"Epoch {epoch+1}: Train Acc = {train_acc:.2f}%, Val Acc = {val_acc:.2f}%")

    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "dress_type_model.pth")

print(f"✅ Training complete! Best validation accuracy: {best_acc:.2f}%")


ModuleNotFoundError: No module named 'torch'

In [ ]:
from PIL import Image
import torchvision.transforms as T

# Load model + labels
model.load_state_dict(torch.load("dress_type_model.pth"))
model.eval()

classes = train_data.classes  # from your dataset

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
])

img = Image.open("test_image.jpg").convert("RGB")
inp = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(inp)
    _, pred = torch.max(output, 1)
    print(f"Predicted Dress Type: {classes[pred.item()]}")


In [11]:
# ----------------------------
# Training MobileNetV3 Small (CPU Optimized)
# ----------------------------

import os
os.environ["OMP_NUM_THREADS"] = "4"

# =========================================================
# System & Performance Settings
# =========================================================
import torch
torch.set_num_threads(4)
# torch.set_num_interop_threads(2)

import datetime
from tqdm import tqdm
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

# =========================================================
# Dataset Paths
# =========================================================
train_dir = "model/dataset-splitted/train"
val_dir = "model/dataset-splitted/val"

# =========================================================
# Data Transforms (smaller image size for speed)
# =========================================================
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((128, 128)),  # smaller size = faster
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    "val": transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

# =========================================================
# Datasets & Dataloaders
# =========================================================
train_data = datasets.ImageFolder(train_dir, transform=data_transforms["train"])
val_data   = datasets.ImageFolder(val_dir, transform=data_transforms["val"])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=2)

# =========================================================
# Model Setup (MobileNetV3 Small)
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
# Replace last classifier layer
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(train_data.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# =========================================================
# Folder for saving models
# =========================================================
os.makedirs("final-model-test-2", exist_ok=True)

# =========================================================
# Training Loop
# =========================================================
best_acc = 0
epochs = 32

for epoch in range(epochs):
    model.train()
    train_loss, correct, total = 0, 0, 0

    print(f"\nEpoch {epoch+1}/{epochs}")
    for imgs, labels in tqdm(train_loader, desc="Training"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total

    # ----------------------------
    # Validation
    # ----------------------------
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="Validation"):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_correct / val_total
    print(f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    # ----------------------------
    # Save best model
    # ----------------------------
    if val_acc > best_acc:
        best_acc = val_acc
        timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        save_path = f"final-model-test-2/dress-type-model-{timestamp}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model: {save_path}")

print(f"\nTraining complete! Best validation accuracy: {best_acc:.2f}%")


Epoch 1/32


Validation: 100%|██████████| 6/6 [00:25<00:00,  4.23s/it]


Train Acc: 5.20% | Val Acc: 8.48%
Saved best model: final-model-test-2/dress-type-model-20251018-212613.pth

Epoch 2/32


Validation: 100%|██████████| 6/6 [00:16<00:00,  2.80s/it]


Train Acc: 12.27% | Val Acc: 11.52%
Saved best model: final-model-test-2/dress-type-model-20251018-212754.pth

Epoch 3/32


...

Train Acc: 94.43% | Val Acc: 32.12%

Epoch 31/32


Validation: 100%|██████████| 6/6 [00:09<00:00,  1.65s/it]


Train Acc: 94.95% | Val Acc: 33.33%

Epoch 32/32


Validation: 100%|██████████| 6/6 [00:09<00:00,  1.59s/it]

Train Acc: 94.95% | Val Acc: 32.12%

Training complete! Best validation accuracy: 36.36%


In [19]:
# ----------------------------
# TEST THE MODEL (MobileNetV3 Small)
# ----------------------------

import torch
from PIL import Image
import torchvision.transforms as T

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the trained model weights
model.load_state_dict(torch.load("final-model-test-2/dress-type-model-20251018-221703.pth", map_location=device))
model.to(device)
model.eval()

# Class labels (reuse from training)
classes = train_data.classes  # Must match your training dataset

# Image transforms (same as training)
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
])

# Load and prepare the image
img_path = "model/abstract-buttoned-top-test.png"  # your test image
img = Image.open(img_path).convert("RGB")
inp = transform(img).unsqueeze(0).to(device)

# Run inference
with torch.no_grad():
    output = model(inp)
    _, pred = torch.max(output, 1)
    predicted_class = classes[pred.item()]

print(f"✅ Predicted Dress Type: {predicted_class}")

✅ Predicted Dress Type: Abstract_Buttoned_Top
